# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 1: Introduction to Natural Language Processing</font>

# <font color="#003660">Notebook 1b: Document Regression</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you will be able to...</b><br><br>
        ... transform raw text into a term-document matrix, and <br>
        ... train a regressor on the term-document matrix.
    </font>
</div>
</center>
</p>

# Import packages

As always, we first need to load a number of required Python packages:
- `pandas` provides high-performance, easy-to-use data structures and data analysis tools.
- `spacy` offers industrial-strength natural language processing.
- `sklearn` is the de-facto standard machine learning package in Python.

In [1]:
#%pip install -q spacy-lookups-data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Load documents

Load 100,000 labeled wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) from a csv file.

In [3]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/wine_reviews_train.csv")

In [4]:
corpus.head()

,id,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [5]:
corpus.shape

(100000, 14)

# Preprocess documents

Split data into a training and a validation set.

In [6]:
training = corpus.iloc[0:80000,].sample(n=10000, random_state=42) # sample to speed up training
validation = corpus.iloc[80000:,]

In [7]:
print(training.shape)
print(validation.shape)

(10000, 14)
(20000, 14)


Perform the same NLP preprocessing steps as in Notebook 1a: tokenize the reviews with spaCy, keep only alphabetic tokens that are not stop words, and reduce them to their lemmas (using a fast lookup lemmatizer).

In [8]:
nlp = spacy.blank("en")
nlp.add_pipe("lemmatizer", config={"mode": "lookup"})
nlp.initialize()

def spacy_prep_df(df, text_col="description", batch_size=1000):
    texts = (str(x).lower() if pd.notna(x) and x else "" for x in df[text_col].values)  # the lookup lemmatizer is case-sensitive
    results = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        toks = [t.lemma_ for t in doc if t.is_alpha and not t.is_stop]
        results.append(" ".join(toks))
    out = df.copy()
    out[text_col + "_prep"] = results
    return out


In [9]:
training = spacy_prep_df(training)

Display the first couple of lines of the preprocessed descriptions.

In [10]:
training["description_prep"].head()

47044    impressive effort cogno offer aroma ripe dark ...
44295    ripe notion orange ripe apple fizz nose palate...
74783    whiff vanilla strawberry entice nose bodied ro...
70975    bottle region northeast san diego austere rest...
46645    stylish wine mineral tannin dusty texture cool...
Name: description_prep, dtype: str

# Vectorize documents

Vectorization is the process of turning a collection of text documents into numerical feature vectors.

We will use the **Bag of Words (BoW)** model for vectorization. In the BoW model, a corpus of documents is represented by a matrix with one row per document and one column per word occurring in the corpus. The cell values will either be simple frequency counts (How often does a word appear in a document?), or the term frequency (tf) times the inverse document frequency (idf) of a term. The idea of tf-idf is to scale down the impact of words that occur very frequently in a given corpus and that are therefore less informative than features that occur only in a small fraction of the corpus. Note that the BoW model completely ignores information about the position and sequences of the words in the document.

In `sklearn`, the [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer) creates a term-document matrix with (normalized) term frequencies and the [`TfIdfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer) creates a term-document matrix with tf-idf weighting.

In [11]:
count_vect = CountVectorizer(min_df=10)

Apply the CountVectorizer object to the review texts of the training set.

In [12]:
X_training = count_vect.fit_transform(training["description_prep"])

Display an extract of the generated term-document matrix

In [13]:
X_training.shape

(10000, 2007)

In [14]:
X_training[0:20,0:20].todense()

matrix([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0,

Store the labels that we want to predict in a separate variable.

In [15]:
y_training = training["points"]
y_training.describe()

count    10000.00000
mean        88.43960
std          3.04284
min         80.00000
25%         86.00000
50%         88.00000
75%         91.00000
max         99.00000
Name: points, dtype: float64

# Train regressor on training set

Fit a linear regression model with the term-document matrix as the features and the wine quality (i.e., `points` variable) as the numerical label.

In [16]:
reg = LinearRegression().fit(X_training, y_training)

Test whether the regression model is working by predicting the quality of a short fake review. We apply the same NLP preprocessing steps and reuse the `count_vect` object to generate features in the same way as we did for the training set.

In [17]:
doc_new = {'description': ['This is a spectacular, magnificent, and majestic wine. Awesome!']}
doc_new_df = pd.DataFrame.from_dict(doc_new)

In [18]:
doc_new_df_prep = spacy_prep_df(doc_new_df)
doc_new_df_prep

,description,description_prep
0,"This is a spectacular, magnificent, and majest...",spectacular magnificent majestic wine awesome


In [19]:
X_new = count_vect.transform(doc_new_df_prep["description_prep"])
predicted = reg.predict(X_new)
predicted

array([84.47543461])

# Evaluate prediction error on validation set

We evaluate the prediction error of our model on the validation set. Again, we apply the same NLP preprocessing steps, reuse the `count_vect` object, and store `X` and `y` in separate data structures.

In [20]:
validation = spacy_prep_df(validation)

In [21]:
X_validation = count_vect.transform(validation["description_prep"])
y_validation = validation["points"]

Call the predict function of our model with the validation data and calculate the mean absolute error (MAE), i.e., by how many points our predictions are off on average.

In [22]:
predictions_validation = reg.predict(X_validation)
print(metrics.mean_absolute_error(y_validation, predictions_validation))

1.5835730132855135


To judge whether this is good or bad, compare it with a naive benchmark that always predicts the mean points of the training set.

In [23]:
print(metrics.mean_absolute_error(y_validation, [y_training.mean()] * len(y_validation)))

2.5655622799999995


Plot predicted vs. actual points for the validation set.

# Model diagnostics

A single number like the MAE tells us *how much* our model is off on average, but not *where* and *why*. Let's take a closer look at the predictions on the validation set. First, we collect true points, predicted points, and the prediction error (predicted minus true points) in one dataframe.

In [ ]:
results = validation[["description", "points"]].copy()
results["predicted"] = predictions_validation
results["error"] = results["predicted"] - results["points"]
print("Predictions outside the 80-100 scale:", ((results["predicted"] < 80) | (results["predicted"] > 100)).sum())
results[["points", "predicted", "error"]].describe()

Plot the true against the predicted points. Each dot is one review (as the true points are integers, we add a little random vertical *jitter* so that the dots don't all lie on top of each other). For a perfect model, all dots would lie on the dashed diagonal. The orange line shows the mean prediction for each true rating.

In [ ]:
rng = np.random.default_rng(42)
jitter = rng.uniform(-0.3, 0.3, len(results))  # spread out the integer ratings a little to reduce overplotting

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(results["predicted"], results["points"] + jitter, s=4, alpha=0.1, color="#2a78d6", label="Single review")
mean_pred = results.groupby("points")["predicted"].mean()
ax.plot(mean_pred.values, mean_pred.index, color="#eb6834", linewidth=2, marker="o", markersize=5, label="Mean prediction per rating")
ax.plot([80, 100], [80, 100], color="gray", linestyle="--", linewidth=1, label="Perfect prediction")
ax.set_yticks(range(80, 101, 2))
ax.set_xlabel("Predicted points")
ax.set_ylabel("True points")
ax.set_title("True vs. predicted points (validation set)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Next, look at the distribution of the prediction errors.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results["error"], bins=60, color="#2a78d6", edgecolor="white")
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Prediction error (predicted - true points)")
ax.set_ylabel("Number of reviews")
ax.set_title("Distribution of prediction errors (validation set)")
ax.grid(alpha=0.3)
plt.show()

How does the error depend on the true quality of a wine? Calculate the mean error (positive = we predict too many points) and the mean absolute error for each true rating.

In [ ]:
results.groupby("points")["error"].agg(["count", "mean", lambda e: e.abs().mean()]).rename(columns={"mean": "mean error", "<lambda_0>": "mean absolute error"}).round(2)

Finally, read the reviews with the largest prediction errors. They often reveal what the model cannot understand.

In [ ]:
results["abs_error"] = results["error"].abs()
results["length"] = results["description"].str.len()  # number of characters
pd.set_option("display.max_colwidth", 300)
results.sort_values("abs_error", ascending=False).head(5)[["points", "predicted", "length", "description"]]

Note that the dataframe display cuts off long texts. Let's print the full text of the review with the largest error and compare its length with a typical review.

In [ ]:
worst = results.sort_values("abs_error", ascending=False).iloc[0]
print(worst["description"])
print()
print("Length of this review:", worst["length"], "characters")
print("Median length of all validation reviews:", results["length"].median(), "characters")

Read it carefully: the first two sentences appear twice, at the beginning and again at the end (probably a copy-paste error in the original data). Let's see what our model predicts when we remove the repeated part, i.e., when we only feed it the text up to the end of the middle part.

In [ ]:
first_part = worst["description"].split("   ")[0]  # the opening sentences (followed by three spaces in the original)
without_repetition = worst["description"][len(first_part):].strip()

def predict_points(text):
    df = spacy_prep_df(pd.DataFrame({"description": [text]}))
    return reg.predict(count_vect.transform(df["description_prep"]))[0]

print("Full text:              ", round(predict_points(worst["description"]), 1))
print("Without the repetition: ", round(predict_points(without_repetition), 1))
print("Only the opening part:  ", round(predict_points(first_part), 1))
print("True points:            ", worst["points"])

Our model adds up the (learned) weight of every word *each time it occurs*. Repeating positive words like "substantial", "complete", and "seamless" therefore inflates the prediction. Is this a general pattern? Let's check how strongly the length of a review is correlated with the predicted and with the true points.

In [ ]:
print("Correlation of text length with predicted points:", round(results["length"].corr(results["predicted"]), 2))
print("Correlation of text length with true points:     ", round(results["length"].corr(results["points"]), 2))

What do the diagnostics tell us?

- **The errors are roughly symmetric and centered at zero**: on average, the model neither over- nor underestimates. About 70% of the predictions are within ±2 points of the true rating.
- **The model is too cautious at the extremes** (*regression to the mean*): the orange line is steeper than the diagonal, i.e., the predictions cover a much narrower range than the true ratings. Mediocre wines (80–83 points) get about 2–4 points too many, outstanding wines (94+ points) about 2 points too few. Extreme ratings are rare in the training data, and many words appear in reviews of all quality levels, so the model hedges towards the average of about 88 points.
- **Linear regression doesn't know the rating scale**: a few predictions fall outside the range of 80–100 points. We could simply clip them to this range.
- **Raw word counts make long texts dangerous**: the model adds up the weight of every word each time it occurs. The worst prediction (above 100 points) belongs to the longest review in the validation set, whose opening sentences are repeated at the end. Without the repetition, the prediction drops considerably. More generally, longer reviews get higher predictions. This is partly justified (reviewers tend to write more about better wines), but the model overshoots for unusually long texts. Normalizing the counts, e.g., with TF-IDF, reduces this effect.
- **Bag-of-words ignores word order**: other reviews start with praise and only turn negative later (e.g., "...will force you to pick up the glass. Unfortunately, ..." – note that the original even misspells *Unfotunately*). A model that only counts words cannot capture such shifts, negations, or the order of words – which is what the neural networks in the next notebooks, and transformers in Session 2, will help with.

Now it's your turn: Can you improve the model? Try, e.g., clipping the predictions, using TF-IDF instead of raw counts (`TfidfVectorizer`), adding bigrams (`ngram_range=(1, 2)`), or a regularized model such as `Ridge`.